In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import qiskit as qsk 
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit
import copy
from collections import deque

type Circs = list[QuantumCircuit]
type Base = int 
type Bases = list[int]
type Bits = list[Bit]
type Bit = 0 | 1

class Upper_Block:
    __i: int = 0
    def __init__(self, indeces: Bits, iter: int):
        self.id = f'BLK_{Upper_Block.__i}'
        Upper_Block.__i += 1
        self.indeces = indeces
        self.iter = iter 
        self.parity = -1

    @classmethod
    def reset(cls):
        cls.__i = 0
    
    def __str__(self) -> str:
        return f'Block: {self.id} | Indeces: {self.indeces} | Iter: {self.iter} | Parity: {self.parity}'



def binary_split( a_bits: Bits, b_bits: Bits, indeces: list[int]) -> int:
    if len(indeces) == 1:
        return indeces[0]
    # recursive break the bit_block into two parts and select the one who presents an odd-error
    break_point = len(indeces) // 2
    left_parity = get_parity(a_bits, b_bits, indeces[:break_point])
    if left_parity % 2 == 1:
        return binary_split(a_bits, b_bits, indeces[:break_point])
    else: 
        return binary_split(a_bits, b_bits, indeces[break_point:])
    


def get_parity(a_bits: Bits, b_bits: Bits, indeces: list[int]) -> int:
    count = 0
    for i in indeces:
        if a_bits[i] != b_bits[i]:
            count += 1
    return count 


def inv_bit(b: Bit) -> Bit:
    return 0 if b == 1 else 1

def block_gen(a_bits: Bits, b_bits: Bits, k: int, ite: int, first_ite_flag: bool = True) -> list[Upper_Block]:
    blocks: list[Upper_Block] = []
    indeces = list(range(len(b_bits)))
    step = 0 
    while step + k < len(b_bits):
        if not first_ite_flag:
            # shuffle the indeces 
            np.random.shuffle(indeces)
        # break down the indeces' list
        # and set the iteration it was made in
        block = Upper_Block(indeces[step:step+k], ite)
        blocks.append(block)
        # set the true parity
        block.parity = get_parity(a_bits, b_bits, block.indeces)

        # proceed with iteration
        step += k 
    
    # last check for a Upper Block of fewer bits
    if step < len(b_bits):
        block = Upper_Block(indeces[step:], ite)
        blocks.append(block)
        block.parity = get_parity(a_bits, b_bits, block.indeces)
    return blocks

def reconcill(a_bits: Bits, b_bits:Bits, Q:float) -> Bits:
    k = int(Q *.73)
    it = 1
    indeces = list(range(len(b_bits)))
    all_blocks: list[Upper_Block] = []
    while k < len(indeces):
        if it != 1:
            np.random.shuffle(indeces)
        blocks = block_gen(a_bits, b_bits, k, it, it == 1)
        all_blocks = [*all_blocks, *blocks]
        k *= 2
        it += 1
    
    for _ in all_blocks:
        print(_)
    # print all blocks before reconcill
    q: deque[Upper_Block] = deque()
    for block in all_blocks:
        # update the parity 
        block.parity = get_parity(a_bits, b_bits, block.indeces)
        q.append(block)
        if block.parity % 2 == 1:
            idx = binary_split(a_bits, b_bits, block.indeces)
            print(f'IDX: {idx}')
            # update the b_bits
            b_bits[idx] = inv_bit(b_bits[idx])
            # update the parity of the block
            block.parity = get_parity(a_bits, b_bits, block.indeces)
            # check the cascading 
            cascade(a_bits, b_bits, q, block.iter)

    for _ in all_blocks:
        print(_)
    return all_blocks

def cascade(a_bits: Bits, b_bits: Bits, q: dict[int, list[Upper_Block]], iter: int):
    keys = list(q.keys())
    keys.reverse()
    keys = [k for k in keys if k < iter]
    print(f'Keys in the cascade for iter {iter} are: {keys}')
    for key in keys:
        blocks = q[key]
        # traverse the blocks 
        for block in blocks:
            block.parity = get_parity(a_bits, b_bits, block.indeces)
            print(f'TRAVERSED BLOCK: {block}')
            if block.parity % 2 == 1:
                # get 
                idx = binary_split(a_bits, b_bits, block.indeces)
                b_bits[idx] = inv_bit(b_bits[idx])
                block.parity = get_parity(a_bits, b_bits, block.indeces)
                print(f'UPDATED BLOCK: {block}')


In [88]:
Q = 6.3
b = [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
a = [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1]

Upper_Block.reset()
blocks = block_gen(a, b, 4, 1)
blocks = [*blocks, *block_gen(a, b, 6, 2, False)]
blocks = [*blocks, *block_gen(a, b, 8, 2, False)]
blocks = [*blocks, *block_gen(a, b, 10, 3, False)]
blocks = [*blocks, *block_gen(a, b, 14, 3, False)]
for _ in blocks:
    print(_)

q: dict[int, list[Upper_Block]] = {}

for _ in blocks:
    _.parity = get_parity(a, b, _.indeces)
    if _.iter in q.keys():
        # append in existing
        q[_.iter].append(_)
    else: 
        # init one
        q[_.iter] = [_]
    if _.parity % 2 == 1:
        idx = binary_split(a, b, _.indeces)
        b[idx] = inv_bit(b[idx])
        _.parity = get_parity(a, b, _.indeces)
        cascade(a, b, q, _.iter)

print(a)
print(b)

Block: BLK_0 | Indeces: [0, 1, 2, 3] | Iter: 1 | Parity: 0
Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Parity: 3
Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Parity: 1
Block: BLK_3 | Indeces: [12, 13, 14, 15] | Iter: 1 | Parity: 0
Block: BLK_4 | Indeces: [16, 17, 18, 19] | Iter: 1 | Parity: 3
Block: BLK_5 | Indeces: [2, 17, 11, 4, 15, 12] | Iter: 2 | Parity: 2
Block: BLK_6 | Indeces: [17, 5, 13, 3, 14, 11] | Iter: 2 | Parity: 2
Block: BLK_7 | Indeces: [9, 4, 18, 2, 11, 7] | Iter: 2 | Parity: 4
Block: BLK_8 | Indeces: [10, 16] | Iter: 2 | Parity: 1
Block: BLK_9 | Indeces: [2, 5, 1, 9, 4, 8, 7, 18] | Iter: 2 | Parity: 5
Block: BLK_10 | Indeces: [14, 8, 17, 0, 15, 18, 12, 1] | Iter: 2 | Parity: 2
Block: BLK_11 | Indeces: [6, 2, 10, 11] | Iter: 2 | Parity: 0
Block: BLK_12 | Indeces: [1, 3, 16, 6, 17, 10, 18, 8, 4, 5] | Iter: 3 | Parity: 5
Block: BLK_13 | Indeces: [12, 2, 19, 14, 0, 11, 15, 13, 7, 9] | Iter: 3 | Parity: 2
Block: BLK_14 | Indeces: [7, 1, 16, 8, 5, 3, 4, 11, 9, 15,

In [89]:
print(len(q))

for block in blocks:
    print(block)

3
Block: BLK_0 | Indeces: [0, 1, 2, 3] | Iter: 1 | Parity: 0
Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Parity: 0
Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Parity: 0
Block: BLK_3 | Indeces: [12, 13, 14, 15] | Iter: 1 | Parity: 0
Block: BLK_4 | Indeces: [16, 17, 18, 19] | Iter: 1 | Parity: 0
Block: BLK_5 | Indeces: [2, 17, 11, 4, 15, 12] | Iter: 2 | Parity: 2
Block: BLK_6 | Indeces: [17, 5, 13, 3, 14, 11] | Iter: 2 | Parity: 2
Block: BLK_7 | Indeces: [9, 4, 18, 2, 11, 7] | Iter: 2 | Parity: 0
Block: BLK_8 | Indeces: [10, 16] | Iter: 2 | Parity: 0
Block: BLK_9 | Indeces: [2, 5, 1, 9, 4, 8, 7, 18] | Iter: 2 | Parity: 0
Block: BLK_10 | Indeces: [14, 8, 17, 0, 15, 18, 12, 1] | Iter: 2 | Parity: 0
Block: BLK_11 | Indeces: [6, 2, 10, 11] | Iter: 2 | Parity: 0
Block: BLK_12 | Indeces: [1, 3, 16, 6, 17, 10, 18, 8, 4, 5] | Iter: 3 | Parity: 0
Block: BLK_13 | Indeces: [12, 2, 19, 14, 0, 11, 15, 13, 7, 9] | Iter: 3 | Parity: 0
Block: BLK_14 | Indeces: [7, 1, 16, 8, 5, 3, 4, 11, 9, 1

In [44]:
blocks[1].parity = get_parity(a, b, blocks[1].indeces)
print(blocks[1])

Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Parity: 0


In [118]:
# for running 
Q = 6.3
b = [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
a = [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1]

Upper_Block.reset()
for _ in block_gen(a, b, 1, 1):
    print(_)
    if _.parity % 2 == 1:
        print(f'Idx;: {binary_split(a, b, _.indeces)}')
    
   

Block: BLK_0 | Indeces: [0] | Iter: 1 | Parity: 0
Block: BLK_1 | Indeces: [1] | Iter: 1 | Parity: 0
Block: BLK_2 | Indeces: [2] | Iter: 1 | Parity: 0
Block: BLK_3 | Indeces: [3] | Iter: 1 | Parity: 0
Block: BLK_4 | Indeces: [4] | Iter: 1 | Parity: 1
Idx;: 4
Block: BLK_5 | Indeces: [5] | Iter: 1 | Parity: 1
Idx;: 5
Block: BLK_6 | Indeces: [6] | Iter: 1 | Parity: 0
Block: BLK_7 | Indeces: [7] | Iter: 1 | Parity: 1
Idx;: 7
Block: BLK_8 | Indeces: [8] | Iter: 1 | Parity: 0
Block: BLK_9 | Indeces: [9] | Iter: 1 | Parity: 1
Idx;: 9
Block: BLK_10 | Indeces: [10] | Iter: 1 | Parity: 0
Block: BLK_11 | Indeces: [11] | Iter: 1 | Parity: 0
Block: BLK_12 | Indeces: [12] | Iter: 1 | Parity: 0
Block: BLK_13 | Indeces: [13] | Iter: 1 | Parity: 0
Block: BLK_14 | Indeces: [14] | Iter: 1 | Parity: 0
Block: BLK_15 | Indeces: [15] | Iter: 1 | Parity: 0
Block: BLK_16 | Indeces: [16] | Iter: 1 | Parity: 1
Idx;: 16
Block: BLK_17 | Indeces: [17] | Iter: 1 | Parity: 1
Idx;: 17
Block: BLK_18 | Indeces: [18] | It

In [133]:
# for running 
Q = 6.3
b = [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
a = [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1]

Upper_Block.reset()
for _ in reconcill(a, b, Q):
    # print(_)
    pass
      
print(b)

Block: BLK_0 | Indeces: [0, 1, 2, 3] | Iter: 1 | Parity: 0
Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Parity: 3
Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Parity: 1
Block: BLK_3 | Indeces: [12, 13, 14, 15] | Iter: 1 | Parity: 0
Block: BLK_4 | Indeces: [16, 17, 18, 19] | Iter: 1 | Parity: 3
Block: BLK_5 | Indeces: [15, 13, 7, 17, 8, 19, 10, 12] | Iter: 2 | Parity: 2
Block: BLK_6 | Indeces: [16, 6, 9, 11, 2, 4, 18, 0] | Iter: 2 | Parity: 4
Block: BLK_7 | Indeces: [12, 7, 8, 1] | Iter: 2 | Parity: 1
Block: BLK_8 | Indeces: [8, 5, 13, 2, 4, 9, 3, 6, 0, 17, 1, 10, 19, 16, 18, 14] | Iter: 3 | Parity: 6
Block: BLK_9 | Indeces: [7, 11, 12, 15] | Iter: 3 | Parity: 1
IDX: 7
IDX: 9
IDX: 18
IDX: 17
IDX: 16
Block: BLK_0 | Indeces: [0, 1, 2, 3] | Iter: 1 | Parity: 0
Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Parity: 2
Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Parity: 0
Block: BLK_3 | Indeces: [12, 13, 14, 15] | Iter: 1 | Parity: 0
Block: BLK_4 | Indeces: [16, 17, 18, 19]